# Groq client

#### 1. Dependencies

In [1]:
%pip install --quiet groq pandas tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: C:\Users\estep\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import pandas as pd
from groq import Groq
from tqdm import tqdm

#### 2. Configuration

In [3]:
MODEL_NAME = "llama-3.3-70b-versatile"
MODEL_SHORT = "Llama-3.3-70B-Groq"

def load_api_key(path, name):
    for line in open(path).read().strip().splitlines():
        if line.startswith(f"{name}="):
            return line.split("=", 1)[1].strip()
    raise KeyError(f"Klucz '{name}' nie znaleziony w {path}")

API_KEY = load_api_key("./api_key.txt", "groq")

PROMPTS_PATH = "./prompts.tsv"
OUTPUT_DIR = f"./responses_{MODEL_SHORT}"
RESPONSES_PATH = f"{OUTPUT_DIR}/responses.tsv"

POLITE_DELAY = 30.0
MAX_OUTPUT_TOKENS = 8192

TARGET_LANGUAGES = ["zh", "fi", "fr", "he", "ja", "pl"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

#### 3. Load prompts

In [4]:
prompts_df = pd.read_csv(PROMPTS_PATH, sep="\t")

try:
    df = pd.read_csv(RESPONSES_PATH, sep="\t")
    done = df["response"].notna().sum()
    print(f"Wznowiono: {done}/{len(df)} wykonanych")
except FileNotFoundError:
    df = prompts_df.copy()
    df["response"] = pd.NA
    print(f"Start od poczatku: {len(df)} promptow do wyslania")

Wznowiono: 0/125 wykonanych


#### 4. Send loop 

In [ ]:
client = Groq(api_key=API_KEY)

MAX_RETRIES = 4
RETRY_BASE_DELAY = 15.0

def is_transient(exc):
    msg = str(exc).upper()
    return any(kw in msg for kw in ("429", "503", "RATE LIMIT", "OVERLOAD", "TRY AGAIN", "UNAVAILABLE", "TIMEOUT"))

def send_one(prompt):
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=MAX_OUTPUT_TOKENS,
            )
            return response.choices[0].message.content or ""
        except Exception as exc:
            if is_transient(exc) and attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_BASE_DELAY * (2 ** attempt))
            else:
                raise

pending = df[df["response"].isna()].index.tolist()
print(f"Do wyslania: {len(pending)} (delay={POLITE_DELAY}s)")

failed = 0
for i in tqdm(pending, desc="Wysylanie"):
    time.sleep(POLITE_DELAY)
    try:
        df.at[i, "response"] = send_one(df.at[i, "prompt"])
    except Exception:
        df.at[i, "response"] = pd.NA
        failed += 1
    df.to_csv(RESPONSES_PATH, sep="\t", index=False)

print(f"Sukces: {len(pending) - failed}, blad: {failed}")

Do wyslania: 125 (delay=30.0s)


Wysylanie:   4%|▍         | 5/125 [11:23<4:33:18, 136.65s/it]

In [ ]:

df = pd.read_csv(RESPONSES_PATH, sep="\t")


In [ ]:
df

,id,slug,target_lang,target_lang_name,prompt,response
0,zh--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,zh,Chinese,You are a professional translator. Translate t...,我知道你们都有很长的待办事项清单，但我讨厌浪费时间，所以我有一个不做清单。不要在社交媒体上滚...
1,fi--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,fi,Finnish,You are a professional translator. Translate t...,"Tiedän, että teillä kaikilla on pitkät tehtävä..."
2,fr--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,fr,French,You are a professional translator. Translate t...,Je sais que vous avez tous des listes de tâche...
3,he--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,he,Hebrew,You are a professional translator. Translate t...,אני יודע שלכולכם יש רשימות ארוכות של דברים שצר...
4,ja--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,ja,Japanese,You are a professional translator. Translate t...,あなたたちは誰しも忙しい生活を送っているので、やることのリストを持っていることでしょう。但し...
5,pl--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,pl,Polish,You are a professional translator. Translate t...,"Wiemy, że wszyscy macie długie listy rzeczy do..."
6,zh--molly_wright_how_every_child_can_thrive_by...,molly_wright_how_every_child_can_thrive_by_five,zh,Chinese,You are a professional translator. Translate t...,[宝宝的啁啾声]\n要是如果我告诉你，一场捉迷藏游戏可以改变世界，你会觉得不可能，对吧？好吧...
7,fi--molly_wright_how_every_child_can_thrive_by...,molly_wright_how_every_child_can_thrive_by_five,fi,Finnish,You are a professional translator. Translate t...,"[Vauvankukkuus]\nMikä jos minä kertoisin, että..."
8,fr--molly_wright_how_every_child_can_thrive_by...,molly_wright_how_every_child_can_thrive_by_five,fr,French,You are a professional translator. Translate t...,[Bébé gazouille]\nEt si je vous disais qu’un j...
9,he--molly_wright_how_every_child_can_thrive_by...,molly_wright_how_every_child_can_thrive_by_five,he,Hebrew,You are a professional translator. Translate t...,[תינוק מישיש]\nמה אם הייתי אומרת לכם שמשחק של ...
